# Sports Betting Agent

The goal of this notebook is to create an agent that can give you explanation and reasoning on the high confidence bets for a given week.

The first run through code will be a demo w/ hard coded values

In [5]:
# ============================================================
# setup.py - Run this once to set up your environment
# ============================================================

import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')

# Verify they loaded
print(f"✓ API Key loaded: {ANTHROPIC_API_KEY[:20]}...")
print(f"✓ GitHub Token loaded: {GITHUB_TOKEN[:20]}...")

✓ API Key loaded: sk-ant-api03-JSz4_ZJ...
✓ GitHub Token loaded: ghp_SrIAMizvgosgOztf...


In [6]:
from typing import Any
from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent
from llama_index.llms.anthropic import Anthropic
from llama_index.core.agent.workflow import AgentStream
from llama_index.core.workflow import Context


# Initialize Claude LLM
llm = Anthropic(api_key=ANTHROPIC_API_KEY, model="claude-haiku-4-5-20251001", max_tokens=2048)
print("✓ Claude LLM initialized")

✓ Claude LLM initialized


hard coded values 

In [7]:
# ============================================================
# Part 1: Define Tool #1 - Model Predictions (REAL DATA)
# ============================================================
import pandas as pd

# For demo purposes - pretend Week 10 2025 is current week
DEMO_WEEK = 10
DEMO_SEASON = 2025

# Load your real predictions tracker
tracker = pd.read_csv('predictions_tracker.csv')

print(f"✓ Tracker loaded: {len(tracker)} total predictions")
print(f"✓ Available weeks: {sorted(tracker['week'].unique())}")

def get_model_predictions(week: int, season: int = 2025) -> dict:
    """
    Fetch real XGBoost model predictions from predictions tracker.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with real game predictions and model edges
    """
    # Filter for requested week
    week_data = tracker[
        (tracker['week'] == week) &
        (tracker['season'] == season)
    ].copy()
    
    if week_data.empty:
        return {
            "week": week,
            "season": season,
            "games": [],
            "message": f"No predictions found for season {season} week {week}"
        }
    
    # Convert to list of game dicts
    games = []
    for _, row in week_data.iterrows():
        games.append({
            "game_id": row['game_id'],
            "matchup": f"{row['home_team']} vs {row['away_team']}",
            "home_team": row['home_team'],
            "away_team": row['away_team'],
            "gameday": str(row['gameday']),
            "spread_line": float(row['spread_line']),
            "predicted_margin": float(row['predicted_margin']),
            "model_edge": float(row['model_edge']),
            "recommendation": row['recommendation']
        })
    
    # Sort by absolute edge (highest edge first)
    games = sorted(games, key=lambda x: abs(x['model_edge']), reverse=True)
    
    # Calculate week summary stats
    edges = [abs(g['model_edge']) for g in games]
    high_edge = len([e for e in edges if e >= 3])
    medium_edge = len([e for e in edges if 1 <= e < 3])
    low_edge = len([e for e in edges if e < 1])
    
    return {
        "week": week,
        "season": season,
        "total_games": len(games),
        "week_summary": {
            "high_edge_games": high_edge,
            "medium_edge_games": medium_edge,
            "low_edge_games": low_edge,
            "avg_edge": round(sum(edges) / len(edges), 2)
        },
        "games": games
    }

# Convert to LlamaIndex tool
prediction_tool = FunctionTool.from_defaults(get_model_predictions)
print("✓ Prediction tool created")

# ============================================================
# Test with real Week 10 2025 data
# ============================================================
test = get_model_predictions(week=DEMO_WEEK, season=DEMO_SEASON)

print(f"\nWeek {DEMO_WEEK} {DEMO_SEASON} Predictions:")
print(f"Total games: {test['total_games']}")
print(f"High edge (3+): {test['week_summary']['high_edge_games']}")
print(f"Medium edge (1-3): {test['week_summary']['medium_edge_games']}")
print(f"Low edge (<1): {test['week_summary']['low_edge_games']}")
print(f"Average edge: {test['week_summary']['avg_edge']}")
print(f"\nGames ranked by edge:")
for g in test['games']:
    print(f"  {g['matchup']}: edge={g['model_edge']} | {g['recommendation']}")

✓ Tracker loaded: 121 total predictions
✓ Available weeks: [np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
✓ Prediction tool created

Week 10 2025 Predictions:
Total games: 14
High edge (3+): 1
Medium edge (1-3): 8
Low edge (<1): 5
Average edge: 1.5

Games ranked by edge:
  TB vs NE: edge=-3.9 | BET AWAY (NE)
  LAC vs PIT: edge=-2.8 | BET AWAY (PIT)
  GB vs PHI: edge=-2.4 | BET AWAY (PHI)
  SF vs LA: edge=-2.3 | BET AWAY (LA)
  CAR vs NO: edge=-1.8 | BET AWAY (NO)
  HOU vs JAX: edge=1.5 | BET HOME (HOU)
  MIN vs BAL: edge=-1.5 | BET AWAY (BAL)
  DEN vs LV: edge=-1.0 | BET AWAY (LV)
  IND vs ATL: edge=-1.0 | BET AWAY (ATL)
  SEA vs ARI: edge=-0.9 | BET AWAY (ARI)
  MIA vs BUF: edge=0.6 | BET HOME (MIA)
  NYJ vs CLE: edge=0.5 | BET HOME (NYJ)
  CHI vs NYG: edge=-0.4 | BET AWAY (NYG)
  WAS vs DET: edge=0.4 | BET HOME (WAS)


In [8]:
# ============================================================
# Part 2: Define Tool #2 - Injury Reports (MOCK - Week 10 2025 Teams)
# ============================================================

# Realistic mock injuries for Week 10 2025 teams
# These are plausible injuries for these matchups
WEEK_10_INJURIES = {
    "DEN": [
        {"player": "Courtland Sutton", "position": "WR", "severity": "questionable", "impact": "medium"}
    ],
    "LV": [
        {"player": "Davante Adams", "position": "WR", "severity": "out", "impact": "high"}
    ],
    "IND": [
        {"player": "Anthony Richardson", "position": "QB", "severity": "questionable", "impact": "high"}
    ],
    "ATL": [
        {"player": "Kyle Pitts", "position": "TE", "severity": "questionable", "impact": "medium"}
    ],
    "CAR": [
        {"player": "Bryce Young", "position": "QB", "severity": "out", "impact": "high"}
    ],
    "NO": [
        {"player": "Derek Carr", "position": "QB", "severity": "questionable", "impact": "high"}
    ],
    "CHI": [
        {"player": "Caleb Williams", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "NYG": [
        {"player": "Daniel Jones", "position": "QB", "severity": "out", "impact": "high"}
    ],
    "HOU": [
        {"player": "CJ Stroud", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "JAX": [
        {"player": "Trevor Lawrence", "position": "QB", "severity": "questionable", "impact": "high"}
    ],
    "MIA": [
        {"player": "Tua Tagovailoa", "position": "QB", "severity": "questionable", "impact": "high"}
    ],
    "BUF": [
        {"player": "Stefon Diggs", "position": "WR", "severity": "out", "impact": "medium"}
    ],
    "MIN": [
        {"player": "Justin Jefferson", "position": "WR", "severity": "probable", "impact": "low"}
    ],
    "BAL": [
        {"player": "Lamar Jackson", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "NYJ": [
        {"player": "Aaron Rodgers", "position": "QB", "severity": "questionable", "impact": "high"}
    ],
    "CLE": [
        {"player": "Deshaun Watson", "position": "QB", "severity": "out", "impact": "high"}
    ],
    "TB": [
        {"player": "Baker Mayfield", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "NE": [
        {"player": "Jacoby Brissett", "position": "QB", "severity": "questionable", "impact": "medium"}
    ],
    "SEA": [
        {"player": "DK Metcalf", "position": "WR", "severity": "questionable", "impact": "medium"}
    ],
    "ARI": [
        {"player": "Kyler Murray", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "SF": [
        {"player": "Christian McCaffrey", "position": "RB", "severity": "out", "impact": "high"}
    ],
    "LA": [
        {"player": "Puka Nacua", "position": "WR", "severity": "questionable", "impact": "medium"}
    ],
    "WAS": [
        {"player": "Jayden Daniels", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "DET": [
        {"player": "Amon-Ra St. Brown", "position": "WR", "severity": "questionable", "impact": "medium"}
    ],
    "LAC": [
        {"player": "Justin Herbert", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "PIT": [
        {"player": "TJ Watt", "position": "LB", "severity": "questionable", "impact": "high"}
    ],
    "GB": [
        {"player": "Jordan Love", "position": "QB", "severity": "probable", "impact": "low"}
    ],
    "PHI": [
        {"player": "AJ Brown", "position": "WR", "severity": "out", "impact": "high"}
    ]
}

def get_injury_reports(week: int, season: int = 2025) -> dict:
    """
    Fetch NFL injury reports for the given week.
    For demo purposes uses realistic mock data for Week 10 2025 teams.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with injured players by team and their impact level
    """
    # Get the teams playing this week from tracker
    week_data = tracker[
        (tracker['week'] == week) &
        (tracker['season'] == season)
    ]
    
    if week_data.empty:
        return {
            "week": week,
            "season": season,
            "injuries": [],
            "message": f"No games found for week {week} season {season}"
        }
    
    # Get all teams playing this week
    home_teams = week_data['home_team'].tolist()
    away_teams = week_data['away_team'].tolist()
    all_teams = home_teams + away_teams
    
    # Build injury list for teams playing this week
    injuries = []
    for team in all_teams:
        if team in WEEK_10_INJURIES:
            for injury in WEEK_10_INJURIES[team]:
                injuries.append({
                    "team": team,
                    "player": injury['player'],
                    "position": injury['position'],
                    "severity": injury['severity'],
                    "impact": injury['impact']
                })
    
    # Sort by impact level
    impact_order = {"high": 0, "medium": 1, "low": 2}
    injuries = sorted(injuries, key=lambda x: impact_order.get(x['impact'], 3))
    
    # Summary stats
    high_impact = len([i for i in injuries if i['impact'] == 'high'])
    medium_impact = len([i for i in injuries if i['impact'] == 'medium'])
    low_impact = len([i for i in injuries if i['impact'] == 'low'])
    
    return {
        "week": week,
        "season": season,
        "total_injuries": len(injuries),
        "summary": {
            "high_impact": high_impact,
            "medium_impact": medium_impact,
            "low_impact": low_impact
        },
        "injuries": injuries
    }

# Convert to LlamaIndex tool
injury_tool = FunctionTool.from_defaults(get_injury_reports)
print("✓ Injury reports tool created")

# ============================================================
# Test with real Week 10 2025 matchups
# ============================================================
test_injuries = get_injury_reports(week=DEMO_WEEK, season=DEMO_SEASON)

print(f"\nWeek {DEMO_WEEK} {DEMO_SEASON} Injury Report:")
print(f"Total injuries: {test_injuries['total_injuries']}")
print(f"High impact: {test_injuries['summary']['high_impact']}")
print(f"Medium impact: {test_injuries['summary']['medium_impact']}")
print(f"Low impact: {test_injuries['summary']['low_impact']}")
print(f"\nInjuries by impact:")
for injury in test_injuries['injuries']:
    print(f"  [{injury['impact'].upper()}] {injury['team']}: {injury['player']} ({injury['position']}) - {injury['severity']}")

✓ Injury reports tool created

Week 10 2025 Injury Report:
Total injuries: 28
High impact: 12
Medium impact: 7
Low impact: 9

Injuries by impact:
  [HIGH] IND: Anthony Richardson (QB) - questionable
  [HIGH] CAR: Bryce Young (QB) - out
  [HIGH] MIA: Tua Tagovailoa (QB) - questionable
  [HIGH] NYJ: Aaron Rodgers (QB) - questionable
  [HIGH] SF: Christian McCaffrey (RB) - out
  [HIGH] LV: Davante Adams (WR) - out
  [HIGH] NO: Derek Carr (QB) - questionable
  [HIGH] NYG: Daniel Jones (QB) - out
  [HIGH] JAX: Trevor Lawrence (QB) - questionable
  [HIGH] CLE: Deshaun Watson (QB) - out
  [HIGH] PIT: TJ Watt (LB) - questionable
  [HIGH] PHI: AJ Brown (WR) - out
  [MEDIUM] DEN: Courtland Sutton (WR) - questionable
  [MEDIUM] SEA: DK Metcalf (WR) - questionable
  [MEDIUM] ATL: Kyle Pitts (TE) - questionable
  [MEDIUM] BUF: Stefon Diggs (WR) - out
  [MEDIUM] NE: Jacoby Brissett (QB) - questionable
  [MEDIUM] LA: Puka Nacua (WR) - questionable
  [MEDIUM] DET: Amon-Ra St. Brown (WR) - questionable

In [9]:
# ============================================================
# Part 3: Define Tool #3 - Line Movement (MOCK - Week 10 2025)
# ============================================================

# Realistic mock line movement for Week 10 2025 matchups
WEEK_10_LINES = {
    "LV_DEN": {
        "home_team": "DEN",
        "away_team": "LV",
        "opening_line": 8.5,
        "current_line": 9.5,
        "movement": "up_1.0",
        "public_percentage": 0.72,
        "sharp_percentage": 0.61,
        "notes": "Sharp money backing DEN, public also on DEN. Line moved in DEN's favor."
    },
    "ATL_IND": {
        "home_team": "IND",
        "away_team": "ATL",
        "opening_line": 5.5,
        "current_line": 6.5,
        "movement": "up_1.0",
        "public_percentage": 0.58,
        "sharp_percentage": 0.52,
        "notes": "Moderate public action on IND. Sharp money slightly on IND."
    },
    "NO_CAR": {
        "home_team": "CAR",
        "away_team": "NO",
        "opening_line": 4.5,
        "current_line": 5.5,
        "movement": "up_1.0",
        "public_percentage": 0.65,
        "sharp_percentage": 0.70,
        "notes": "Sharp money heavily backing CAR. QB injury concerns for NO."
    },
    "NYG_CHI": {
        "home_team": "CHI",
        "away_team": "NYG",
        "opening_line": 3.5,
        "current_line": 4.5,
        "movement": "up_1.0",
        "public_percentage": 0.55,
        "sharp_percentage": 0.48,
        "notes": "Public on CHI but sharp money fading slightly. Mixed signals."
    },
    "JAX_HOU": {
        "home_team": "HOU",
        "away_team": "JAX",
        "opening_line": 6.5,
        "current_line": 7.5,
        "movement": "up_1.0",
        "public_percentage": 0.68,
        "sharp_percentage": 0.65,
        "notes": "Both public and sharp money on HOU. Strong consensus."
    },
    "BUF_MIA": {
        "home_team": "MIA",
        "away_team": "BUF",
        "opening_line": -2.5,
        "current_line": -1.5,
        "movement": "down_1.0",
        "public_percentage": 0.62,
        "sharp_percentage": 0.38,
        "notes": "Sharp money fading MIA despite public support. Line moving against MIA."
    },
    "BAL_MIN": {
        "home_team": "MIN",
        "away_team": "BAL",
        "opening_line": -3.5,
        "current_line": -2.5,
        "movement": "down_1.0",
        "public_percentage": 0.44,
        "sharp_percentage": 0.68,
        "notes": "Sharp money heavily on BAL. Line moving in BAL's favor despite public on MIN."
    },
    "CLE_NYJ": {
        "home_team": "NYJ",
        "away_team": "CLE",
        "opening_line": 2.5,
        "current_line": 3.0,
        "movement": "up_0.5",
        "public_percentage": 0.55,
        "sharp_percentage": 0.50,
        "notes": "Slight public edge on NYJ. Sharp money split. Low conviction week."
    },
    "NE_TB": {
        "home_team": "TB",
        "away_team": "NE",
        "opening_line": -6.5,
        "current_line": -5.5,
        "movement": "down_1.0",
        "public_percentage": 0.38,
        "sharp_percentage": 0.72,
        "notes": "Sharp money heavily on NE. Line moving in NE's favor. Fade TB."
    },
    "ARI_SEA": {
        "home_team": "SEA",
        "away_team": "ARI",
        "opening_line": 3.5,
        "current_line": 2.5,
        "movement": "down_1.0",
        "public_percentage": 0.60,
        "sharp_percentage": 0.45,
        "notes": "Public on SEA but sharp money on ARI. Conflicting signals."
    },
    "LA_SF": {
        "home_team": "SF",
        "away_team": "LA",
        "opening_line": -4.5,
        "current_line": -3.5,
        "movement": "down_1.0",
        "public_percentage": 0.42,
        "sharp_percentage": 0.65,
        "notes": "Sharp money on LA despite SF being home favorite. McCaffrey injury impact."
    },
    "DET_WAS": {
        "home_team": "WAS",
        "away_team": "DET",
        "opening_line": -3.5,
        "current_line": -2.5,
        "movement": "down_1.0",
        "public_percentage": 0.48,
        "sharp_percentage": 0.58,
        "notes": "Sharp money slightly on DET. Public split. Low conviction game."
    },
    "PIT_LAC": {
        "home_team": "LAC",
        "away_team": "PIT",
        "opening_line": -4.5,
        "current_line": -3.5,
        "movement": "down_1.0",
        "public_percentage": 0.52,
        "sharp_percentage": 0.62,
        "notes": "Sharp money on PIT. TJ Watt injury concern moving the line."
    },
    "PHI_GB": {
        "home_team": "GB",
        "away_team": "PHI",
        "opening_line": -5.5,
        "current_line": -6.5,
        "movement": "up_1.0",
        "public_percentage": 0.70,
        "sharp_percentage": 0.68,
        "notes": "Both public and sharp money on PHI. AJ Brown injury could be a concern."
    }
}

def get_line_movement(week: int, season: int = 2025) -> dict:
    """
    Fetch Vegas line movement data for the week.
    Shows opening line, current line, and public vs sharp action.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with line movement data for each game
    """
    # Get the teams playing this week from tracker
    week_data = tracker[
        (tracker['week'] == week) &
        (tracker['season'] == season)
    ]
    
    if week_data.empty:
        return {
            "week": week,
            "season": season,
            "games": [],
            "message": f"No games found for week {week} season {season}"
        }
    
    # Build line movement for each game
    games = []
    for _, row in week_data.iterrows():
        home = row['home_team']
        away = row['away_team']
        
        # Try both key orders
        key = f"{away}_{home}" if f"{away}_{home}" in WEEK_10_LINES else f"{home}_{away}"
        
        if key in WEEK_10_LINES:
            line_data = WEEK_10_LINES[key]
            games.append({
                "game_id": row['game_id'],
                "matchup": f"{home} vs {away}",
                "home_team": home,
                "away_team": away,
                "opening_line": line_data['opening_line'],
                "current_line": line_data['current_line'],
                "movement": line_data['movement'],
                "public_percentage": line_data['public_percentage'],
                "sharp_percentage": line_data['sharp_percentage'],
                "sharp_vs_public": "SHARP FAVORED" if line_data['sharp_percentage'] > line_data['public_percentage'] else "PUBLIC FAVORED",
                "notes": line_data['notes']
            })
        else:
            # Fallback if game not found
            games.append({
                "game_id": row['game_id'],
                "matchup": f"{home} vs {away}",
                "home_team": home,
                "away_team": away,
                "opening_line": row['spread_line'],
                "current_line": row['spread_line'],
                "movement": "none",
                "public_percentage": 0.50,
                "sharp_percentage": 0.50,
                "sharp_vs_public": "NO DATA",
                "notes": "No line movement data available for this game"
            })
    
    # Summary stats
    sharp_games = len([g for g in games if g['sharp_vs_public'] == "SHARP FAVORED"])
    public_games = len([g for g in games if g['sharp_vs_public'] == "PUBLIC FAVORED"])
    
    return {
        "week": week,
        "season": season,
        "total_games": len(games),
        "summary": {
            "sharp_favored_games": sharp_games,
            "public_favored_games": public_games
        },
        "games": games
    }

# Convert to LlamaIndex tool
line_tool = FunctionTool.from_defaults(get_line_movement)
print("✓ Line movement tool created")

# ============================================================
# Test with real Week 10 2025 matchups
# ============================================================
test_lines = get_line_movement(week=DEMO_WEEK, season=DEMO_SEASON)

print(f"\nWeek {DEMO_WEEK} {DEMO_SEASON} Line Movement:")
print(f"Total games: {test_lines['total_games']}")
print(f"Sharp favored: {test_lines['summary']['sharp_favored_games']}")
print(f"Public favored: {test_lines['summary']['public_favored_games']}")
print(f"\nLine movement by game:")
for g in test_lines['games']:
    print(f"  {g['matchup']}:")
    print(f"    Opening: {g['opening_line']} → Current: {g['current_line']} ({g['movement']})")
    print(f"    Public: {g['public_percentage']*100:.0f}% | Sharp: {g['sharp_percentage']*100:.0f}% | {g['sharp_vs_public']}")
    print(f"    Notes: {g['notes']}")
    print()

✓ Line movement tool created

Week 10 2025 Line Movement:
Total games: 14
Sharp favored: 6
Public favored: 8

Line movement by game:
  DEN vs LV:
    Opening: 8.5 → Current: 9.5 (up_1.0)
    Public: 72% | Sharp: 61% | PUBLIC FAVORED
    Notes: Sharp money backing DEN, public also on DEN. Line moved in DEN's favor.

  IND vs ATL:
    Opening: 5.5 → Current: 6.5 (up_1.0)
    Public: 58% | Sharp: 52% | PUBLIC FAVORED
    Notes: Moderate public action on IND. Sharp money slightly on IND.

  CAR vs NO:
    Opening: 4.5 → Current: 5.5 (up_1.0)
    Public: 65% | Sharp: 70% | SHARP FAVORED
    Notes: Sharp money heavily backing CAR. QB injury concerns for NO.

  CHI vs NYG:
    Opening: 3.5 → Current: 4.5 (up_1.0)
    Public: 55% | Sharp: 48% | PUBLIC FAVORED
    Notes: Public on CHI but sharp money fading slightly. Mixed signals.

  HOU vs JAX:
    Opening: 6.5 → Current: 7.5 (up_1.0)
    Public: 68% | Sharp: 65% | PUBLIC FAVORED
    Notes: Both public and sharp money on HOU. Strong consensus

In [10]:
# ============================================================
# Part 4: Define Tool #4 - Historical Matchup Data
# ============================================================
import nflreadpy as nfl
import pandas as pd

# For demo purposes - pretend Week 10 2025 is current week
DEMO_WEEK = 10
DEMO_SEASON = 2025

print("Loading historical schedule data...")

# Load data up to 2024 (everything before our "current" week)
raw = nfl.load_schedules(list(range(2015, 2025)))
schedule = raw.to_pandas() if hasattr(raw, 'to_pandas') else pd.DataFrame(raw)

# Keep only completed regular season games
schedule = schedule[
    (schedule['game_type'] == 'REG') &
    (schedule['result'].notna())
].copy()

print(f"✓ Loaded {len(schedule)} completed games (2015-2024)")
print(f"✓ Simulating: Week {DEMO_WEEK}, Season {DEMO_SEASON}")

def get_historical_matchup(team1: str, team2: str) -> dict:
    """
    Get real head-to-head historical data between two teams.
    Uses all data available BEFORE Week 10 2025 (our demo current week).
    
    Args:
        team1: Home team abbreviation (e.g., "DEN")
        team2: Away team abbreviation (e.g., "LV")
    
    Returns:
        Dictionary with real head-to-head record and recent game history
    """
    
    # Find all games between these two teams
    matchup_games = schedule[
        ((schedule['home_team'] == team1) & (schedule['away_team'] == team2)) |
        ((schedule['home_team'] == team2) & (schedule['away_team'] == team1))
    ].copy().sort_values(['season', 'week'], ascending=False)
    
    if matchup_games.empty:
        return {
            "team1": team1,
            "team2": team2,
            "head_to_head_record": "No history found",
            "avg_margin": 0,
            "last_5_games": [],
            "notes": f"No historical data found for {team1} vs {team2}"
        }
    
    # Calculate wins for each team
    team1_wins = 0
    team2_wins = 0
    margins = []
    last_5 = []
    
    for _, game in matchup_games.iterrows():
        home = game['home_team']
        result = game['result']  # positive = home team won
        
        # Determine winner and margin from team1's perspective
        if home == team1:
            winner = team1 if result > 0 else team2
            team1_score = game.get('home_score', None)
            team2_score = game.get('away_score', None)
        else:
            winner = team1 if result < 0 else team2
            team1_score = game.get('away_score', None)
            team2_score = game.get('home_score', None)
        
        if winner == team1:
            team1_wins += 1
        else:
            team2_wins += 1
        
        margins.append(abs(result))
        
        # Build last 5 games list
        if len(last_5) < 5:
            last_5.append({
                "date": str(game.get('gameday', 'Unknown')),
                "winner": winner,
                "margin": round(abs(result), 1),
                "team1_score": int(team1_score) if pd.notna(team1_score) else None,
                "team2_score": int(team2_score) if pd.notna(team2_score) else None
            })
    
    total_games = team1_wins + team2_wins
    avg_margin = round(sum(margins) / len(margins), 1) if margins else 0
    recent_wins = sum(1 for g in last_5 if g['winner'] == team1)
    
    # Determine series leader
    if team1_wins > team2_wins:
        record = f"{team1} leads {team1_wins}-{team2_wins}"
        dominant = team1
    elif team2_wins > team1_wins:
        record = f"{team2} leads {team2_wins}-{team1_wins}"
        dominant = team2
    else:
        record = f"Series tied {team1_wins}-{team2_wins}"
        dominant = None
    
    # Generate notes
    if dominant:
        notes = f"{dominant} leads the all-time series {max(team1_wins, team2_wins)}-{min(team1_wins, team2_wins)} since 2015. "
    else:
        notes = "Series is evenly matched since 2015. "
    notes += f"{team1} has won {recent_wins} of the last {len(last_5)} meetings. Average margin: {avg_margin} points."
    
    return {
        "team1": team1,
        "team2": team2,
        "head_to_head_record": record,
        "total_games": total_games,
        "team1_wins": team1_wins,
        "team2_wins": team2_wins,
        "avg_margin": avg_margin,
        "last_5_games": last_5,
        "notes": notes
    }

# Convert to LlamaIndex tool
history_tool = FunctionTool.from_defaults(get_historical_matchup)
print("✓ Historical matchup tool created")

# ============================================================
# Test with REAL Week 10 2025 matchups
# ============================================================
week10 = tracker[tracker['week'] == DEMO_WEEK]

print(f"\nHistorical data for all Week {DEMO_WEEK} {DEMO_SEASON} matchups:\n")
for _, row in week10.iterrows():
    h = get_historical_matchup(row['home_team'], row['away_team'])
    print(f"{row['home_team']} vs {row['away_team']}:")
    print(f"  Record: {h['head_to_head_record']}")
    print(f"  Last 5: {row['home_team']} won {sum(1 for g in h['last_5_games'] if g['winner'] == row['home_team'])} of 5")
    print(f"  Notes: {h['notes']}")
    print()

Loading historical schedule data...
✓ Loaded 2623 completed games (2015-2024)
✓ Simulating: Week 10, Season 2025
✓ Historical matchup tool created

Historical data for all Week 10 2025 matchups:

DEN vs LV:
  Record: LV leads 8-2
  Last 5: DEN won 2 of 5
  Notes: LV leads the all-time series 8-2 since 2015. DEN has won 2 of the last 5 meetings. Average margin: 9.5 points.

IND vs ATL:
  Record: IND leads 2-1
  Last 5: IND won 2 of 5
  Notes: IND leads the all-time series 2-1 since 2015. IND has won 2 of the last 3 meetings. Average margin: 8.3 points.

CAR vs NO:
  Record: NO leads 12-8
  Last 5: CAR won 2 of 5
  Notes: NO leads the all-time series 12-8 since 2015. CAR has won 2 of the last 5 meetings. Average margin: 11.6 points.

CHI vs NYG:
  Record: Series tied 3-3
  Last 5: CHI won 3 of 5
  Notes: Series is evenly matched since 2015. CHI has won 3 of the last 5 meetings. Average margin: 8.7 points.

HOU vs JAX:
  Record: HOU leads 16-4
  Last 5: HOU won 3 of 5
  Notes: HOU leads t

In [11]:
# ============================================================
# Part 5: Define Tool #5 - Model Confidence Analysis (REAL DATA)
# ============================================================

def analyze_model_confidence(week: int, season: int = 2025) -> dict:
    """
    Analyze real model confidence for the given week from predictions tracker.
    Uses actual model edge and prediction data.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with confidence statistics and recommendations
    """
    # Get real week data from tracker
    week_data = tracker[
        (tracker['week'] == week) &
        (tracker['season'] == season)
    ].copy()
    
    if week_data.empty:
        return {
            "week": week,
            "season": season,
            "message": f"No data found for week {week} season {season}"
        }
    
    # Calculate edge statistics
    edges = week_data['model_edge'].abs()
    avg_edge = round(edges.mean(), 2)
    max_edge = round(edges.max(), 2)
    min_edge = round(edges.min(), 2)
    
    # Bucket games by edge
    high_edge_games = week_data[edges >= 3][['home_team', 'away_team', 'model_edge', 'recommendation']].to_dict('records')
    medium_edge_games = week_data[(edges >= 1) & (edges < 3)][['home_team', 'away_team', 'model_edge', 'recommendation']].to_dict('records')
    low_edge_games = week_data[edges < 1][['home_team', 'away_team', 'model_edge', 'recommendation']].to_dict('records')
    
    # Format game entries
    def format_games(games):
        return [
            {
                "matchup": f"{g['home_team']} vs {g['away_team']}",
                "model_edge": g['model_edge'],
                "recommendation": g['recommendation']
            }
            for g in games
        ]
    
    # Determine overall conviction level
    if avg_edge >= 3:
        conviction = "HIGH"
        bet_sizing = "Bet aggressively - high conviction week"
    elif avg_edge >= 1.5:
        conviction = "MEDIUM"
        bet_sizing = "Bet moderately - focus on high edge games only"
    else:
        conviction = "LOW"
        bet_sizing = "Bet conservatively - only consider high edge games"
    
    # Best plays (top 3 by absolute edge)
    best_plays = week_data.nlargest(3, 'model_edge', keep='all')
    worst_plays = week_data.nsmallest(3, 'model_edge', keep='all')
    
    top_bets = []
    for _, row in best_plays.iterrows():
        top_bets.append({
            "matchup": f"{row['home_team']} vs {row['away_team']}",
            "model_edge": row['model_edge'],
            "recommendation": row['recommendation']
        })
    
    skip_games = []
    for _, row in week_data[edges < 1].iterrows():
        skip_games.append({
            "matchup": f"{row['home_team']} vs {row['away_team']}",
            "model_edge": row['model_edge'],
            "recommendation": row['recommendation']
        })
    
    return {
        "week": week,
        "season": season,
        "total_games": len(week_data),
        "edge_statistics": {
            "average_edge": avg_edge,
            "max_edge": max_edge,
            "min_edge": min_edge
        },
        "edge_distribution": {
            "high_edge_count": len(high_edge_games),    # >= 3 points
            "medium_edge_count": len(medium_edge_games), # 1-3 points
            "low_edge_count": len(low_edge_games)        # < 1 point
        },
        "high_edge_games": format_games(high_edge_games),
        "medium_edge_games": format_games(medium_edge_games),
        "low_edge_games": format_games(low_edge_games),
        "conviction_level": conviction,
        "bet_sizing_recommendation": bet_sizing,
        "top_3_bets": top_bets,
        "skip_games": skip_games,
        "overall_recommendation": f"Week {week} is a {conviction} conviction week with average edge of {avg_edge} points. {bet_sizing}."
    }

# Convert to LlamaIndex tool
confidence_tool = FunctionTool.from_defaults(analyze_model_confidence)
print("✓ Model confidence tool created")

# ============================================================
# Test with real Week 10 2025 data
# ============================================================
test_conf = analyze_model_confidence(week=DEMO_WEEK, season=DEMO_SEASON)

print(f"\nWeek {DEMO_WEEK} {DEMO_SEASON} Model Confidence Analysis:")
print(f"Total games: {test_conf['total_games']}")
print(f"\nEdge Statistics:")
print(f"  Average edge: {test_conf['edge_statistics']['average_edge']}")
print(f"  Max edge: {test_conf['edge_statistics']['max_edge']}")
print(f"  Min edge: {test_conf['edge_statistics']['min_edge']}")
print(f"\nEdge Distribution:")
print(f"  High edge (3+): {test_conf['edge_distribution']['high_edge_count']} games")
print(f"  Medium edge (1-3): {test_conf['edge_distribution']['medium_edge_count']} games")
print(f"  Low edge (<1): {test_conf['edge_distribution']['low_edge_count']} games")
print(f"\nConviction Level: {test_conf['conviction_level']}")
print(f"Bet Sizing: {test_conf['bet_sizing_recommendation']}")
print(f"\nTop 3 Bets:")
for bet in test_conf['top_3_bets']:
    print(f"  {bet['matchup']}: edge={bet['model_edge']} | {bet['recommendation']}")
print(f"\nSkip These Games (<1 edge):")
for game in test_conf['skip_games']:
    print(f"  {game['matchup']}: edge={game['model_edge']}")
print(f"\nOverall: {test_conf['overall_recommendation']}")

✓ Model confidence tool created

Week 10 2025 Model Confidence Analysis:
Total games: 14

Edge Statistics:
  Average edge: 1.5
  Max edge: 3.9
  Min edge: 0.4

Edge Distribution:
  High edge (3+): 1 games
  Medium edge (1-3): 8 games
  Low edge (<1): 5 games

Conviction Level: MEDIUM
Bet Sizing: Bet moderately - focus on high edge games only

Top 3 Bets:
  HOU vs JAX: edge=1.5 | BET HOME (HOU)
  MIA vs BUF: edge=0.6 | BET HOME (MIA)
  NYJ vs CLE: edge=0.5 | BET HOME (NYJ)

Skip These Games (<1 edge):
  CHI vs NYG: edge=-0.4
  MIA vs BUF: edge=0.6
  NYJ vs CLE: edge=0.5
  SEA vs ARI: edge=-0.9
  WAS vs DET: edge=0.4

Overall: Week 10 is a MEDIUM conviction week with average edge of 1.5 points. Bet moderately - focus on high edge games only.


In [12]:
# ============================================================
# Part 6: Combine All Tools & Create Agent (FIXED)
# ============================================================

# Step 1: Collect all tools into a list
tools = [
    prediction_tool,
    injury_tool,
    line_tool,
    history_tool,
    confidence_tool
]

print(f"✓ All {len(tools)} tools registered")

agent = ReActAgent(tools=tools, llm=llm, verbose=True, max_iterations=10,
    system_prompt="""You are an expert sports betting analyst with deep knowledge of NFL games.

    Your job is to analyze NFL games and recommend high-conviction bets.

    When analyzing games, you MUST:
    1. First get model predictions for the week
    2. Check injury reports for key players
    3. Review line movement to see smart money vs public
    4. Look up historical matchups between teams
    5. Analyze overall model confidence for the week

    Then synthesize all this information and:
    - Rank games by confidence and edge
    - Explain why you recommend each bet
    - Flag any conflicts (e.g., model says yes but sharp money says no)
    - Give an overall conviction level for the week
    - Recommend bet sizing based on confidence

    Be conservative with low-confidence predictions. Always explain your reasoning."""
)

# Create a context to store the conversation history/session state
ctx = Context(agent)

print("✓ ReActAgent created with system prompt")
print("\nAgent is ready to analyze games!")

✓ All 5 tools registered
✓ ReActAgent created with system prompt

Agent is ready to analyze games!


In [13]:
# ============================================================
# Part 7: Run the Agent
# ============================================================
from llama_index.core.agent.workflow import AgentStream, ToolCallResult

print("="*80)
print("🏈 NFL WEEK 10 2025 BETTING ANALYSIS (BACKTESTING)")
print("="*80)
print()

handler = agent.run(
    f"Analyze NFL season {DEMO_SEASON} week {DEMO_WEEK} games. "
    "Which games should we bet on? "
    "Consider model predictions, injuries, line movement, and historical matchups. "
    "Rank your recommendations by confidence and explain your reasoning for each. "
    "Format your response clearly with sections for: "
    "1) HIGH CONFIDENCE BETS 2) MEDIUM CONFIDENCE BETS 3) SKIP THESE GAMES 4) OVERALL WEEK ASSESSMENT",
    ctx=ctx
)

async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print(f"\n🔧 Tool Called: {ev.tool_name}")
        print(f"   Input: {ev.tool_kwargs}")
        print(f"   {'─'*60}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler

print("\n")
print("="*80)
print("📊 ACTUAL WEEK 10 RESULTS (GROUND TRUTH)")
print("="*80)
week10 = tracker[tracker['week'] == DEMO_WEEK]
for _, row in week10.iterrows():
    correct = "✅" if row['model_correct'] == 1 else "❌"
    print(f"  {correct} {row['home_team']} vs {row['away_team']} | Rec: {row['recommendation']} | Edge: {row['model_edge']}")
print(f"\nModel accuracy: {week10['model_correct'].sum()}/{len(week10)} ({week10['model_correct'].mean()*100:.1f}%)")

🏈 NFL WEEK 10 2025 BETTING ANALYSIS (BACKTESTING)

[tick] add: AgentWorkflowStartEvent(user_msg='Analyze NFL season 2025 week 10 games. Which games...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Analyze NFL season 2025 week 10 games. Which games should we bet on? ...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are an expert sports betting analyst with deep knowledge of N...
[run_agent_step:0] started from AgentSetup
Thought: The user is asking for a comprehensive analysis of NFL Week 10 2025 games with betting recommendations. I need to gathe

In [14]:
# ============================================================
# Format Final Response
# ============================================================

# Extract clean text from response
if hasattr(response, 'response'):
    final_text = response.response.blocks[0].text
elif hasattr(response, 'message'):
    final_text = response.message.content
else:
    final_text = str(response)

print("\n")
print("="*80)
print("📊 FINAL RECOMMENDATION SUMMARY")
print("="*80)
print(final_text)
print("="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)



📊 FINAL RECOMMENDATION SUMMARY
# NFL WEEK 10 2025 BETTING ANALYSIS & RECOMMENDATIONS

## 1) HIGH CONFIDENCE BETS

### 🔥 TIER 1: STRONGEST CONVICTION

**NE at TB (Sunday 11/9)**
- **RECOMMENDATION: BET NEW ENGLAND +2.5**
- **Confidence Level: VERY HIGH**
- **Bet Size: 2-3 units**

**Why This Bet:**
- Model Edge: -3.9 (HIGHEST of the week)
- Sharp Money Alignment: 72% of sharp money on NE vs only 38% public
- Line Movement: Opened at -6.5 TB, now -5.5 TB (moving toward NE)
- Historical Context: Series tied 1-1, evenly matched
- Injury Impact: Both QBs probable (low impact)
- **Conflict Resolution:** Sharp money is HEAVILY backing NE despite public fading. This is a classic sharp vs public spot where the smart money is clearly on NE.

---

**HOU vs JAX (Sunday 11/9)**
- **RECOMMENDATION: BET HOUSTON -1.5**
- **Confidence Level: HIGH**
- **Bet Size: 2 units**

**Why This Bet:**
- Model Edge: +1.5 (only positive edge for home team)
- Sharp Money Alignment: 65% sharp, 68% public (strong co

In [21]:
def parse_analysis_by_game(analysis_text: str, week_data) -> dict:
    """
    Parse agent analysis into per-game sections.
    Returns a dict keyed by matchup string.
    """
    import re
    
    game_analysis = {}
    
    # Get all matchups from tracker
    matchups = []
    for _, row in week_data.iterrows():
        matchups.append({
            "key": f"{row['home_team']}_{row['away_team']}",
            "home": row['home_team'],
            "away": row['away_team']
        })
    
    # Split analysis into lines
    lines = analysis_text.split('\n')
    
    current_game = None
    current_lines = []
    
    for line in lines:
        # Check if this line mentions any team
        matched_game = None
        for matchup in matchups:
            home = matchup['home']
            away = matchup['away']
            # Check if line contains both teams or game reference
            if (home in line and away in line) or \
               (f"{away} vs {home}" in line) or \
               (f"{home} vs {away}" in line) or \
               (f"{away} @ {home}" in line):
                matched_game = matchup['key']
                break
        
        if matched_game:
            # Save previous game's analysis
            if current_game and current_lines:
                game_analysis[current_game] = '\n'.join(current_lines).strip()
            current_game = matched_game
            current_lines = [line]
        elif current_game:
            current_lines.append(line)
    
    # Save last game
    if current_game and current_lines:
        game_analysis[current_game] = '\n'.join(current_lines).strip()
    
    return game_analysis


def save_analysis_to_json(analysis_text: str, week: int, season: int, tracker):
    """Save agent analysis to JSON with per-game breakdown."""
    
    week_data = tracker[
        (tracker['week'] == week) &
        (tracker['season'] == season)
    ]
    
    # Parse into per-game sections
    game_analysis = parse_analysis_by_game(analysis_text, week_data)
    
    # Clean up trailing bullets
    lines = analysis_text.split('\n')
    cleaned_lines = [l for l in lines if l.strip() not in ['•', '-', '*', '·']]
    while cleaned_lines and cleaned_lines[-1].strip() == '':
        cleaned_lines.pop()
    clean_text = '\n'.join(cleaned_lines)
    
    cache_file = f"agent_analysis_{season}_week{week}.json"
    data = {
        "week": week,
        "season": season,
        "analysis": clean_text,           # Full analysis (backup)
        "game_analysis": game_analysis,   # Per-game sections
        "generated_at": dt.now().isoformat()
    }
    
    with open(cache_file, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"✓ Analysis saved to {cache_file}")
    print(f"✓ Per-game sections: {len(game_analysis)}")
    for key, val in game_analysis.items():
        print(f"  - {key}: {len(val)} chars")
    
    return cache_file

# Save the analysis
cache_file = save_analysis_to_json(
    analysis_text=str(response),
    week=DEMO_WEEK,
    season=DEMO_SEASON,
    tracker=tracker
)

✓ Analysis saved to agent_analysis_2025_week10.json
✓ Per-game sections: 11
  - TB_NE: 336 chars
  - HOU_JAX: 183 chars
  - SF_LA: 180 chars
  - MIN_BAL: 236 chars
  - LAC_PIT: 273 chars
  - GB_PHI: 359 chars
  - CAR_NO: 145 chars
  - MIA_BUF: 203 chars
  - NYJ_CLE: 221 chars
  - WAS_DET: 202 chars
  - SEA_ARI: 128 chars


model check code

In [ ]:
'''from anthropic import Anthropic as AnthropicClient

client = AnthropicClient(api_key=ANTHROPIC_API_KEY)

# Try Haiku models
models_to_try = [
    "claude-haiku-4-5-20251001",
    "claude-haiku-3-5-20241022",
    "claude-3-haiku-20240307",
    "claude-haiku-4-0-20250514"
]

for model in models_to_try:
    try:
        msg = client.messages.create(
            model=model,
            max_tokens=10,
            messages=[{"role": "user", "content": "hi"}]
        )
        print(f"✓ {model} works!")
    except Exception as e:
        print(f"✗ {model} failed")'''

'from anthropic import Anthropic as AnthropicClient\n\nclient = AnthropicClient(api_key=ANTHROPIC_API_KEY)\n\n# Try Haiku models\nmodels_to_try = [\n    "claude-haiku-4-5-20251001",\n    "claude-haiku-3-5-20241022",\n    "claude-3-haiku-20240307",\n    "claude-haiku-4-0-20250514"\n]\n\nfor model in models_to_try:\n    try:\n        msg = client.messages.create(\n            model=model,\n            max_tokens=10,\n            messages=[{"role": "user", "content": "hi"}]\n        )\n        print(f"✓ {model} works!")\n    except Exception as e:\n        print(f"✗ {model} failed")'

In [ ]:
'''from llama_index.llms.anthropic import Anthropic

# Initialize with correct model name
llm = Anthropic(api_key=ANTHROPIC_API_KEY, model="claude-haiku-4-5-20251001")
print("✓ Claude LLM initialized")

# Test it
resp = llm.complete("Who is Paul Graham?")
print(resp)'''

'from llama_index.llms.anthropic import Anthropic\n\n# Initialize with correct model name\nllm = Anthropic(api_key=ANTHROPIC_API_KEY, model="claude-haiku-4-5-20251001")\nprint("✓ Claude LLM initialized")\n\n# Test it\nresp = llm.complete("Who is Paul Graham?")\nprint(resp)'